In [1]:
import os
import sys

sys.path.insert(
    0, os.path.abspath("../../")
)

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
root_dir = Path("../../").resolve()
print(">> Root directory:", root_dir)

>> Root directory: /home/hgkahng/Workspaces/soft-prompt


## 1. Load Oracle Data

### 1-1. Load Embeddings

In [3]:
data_dir = root_dir / "data/agnews"
embedding_dir = data_dir / "embeddings/openai/text-embedding-3-small"
print(*sorted(os.listdir(embedding_dir)), sep='\n')

test.features.npy
test.labels.npy
train.features.npy
train.labels.npy


In [4]:
X_train = np.load(embedding_dir / "train.features.npy")
y_train = np.load(embedding_dir / "train.labels.npy")
print(X_train.shape, y_train.shape)

(120000, 1536) (120000,)


In [5]:
X_test = np.load(embedding_dir / "test.features.npy")
y_test = np.load(embedding_dir / "test.labels.npy")
print(X_test.shape, y_test.shape)

(7600, 1536) (7600,)


### 1-2. Load Text

In [6]:
from datasets import load_dataset
ds = load_dataset("SetFit/ag_news", trust_remote_code=True)

In [7]:
df_train = ds['train'].to_pandas()
df_test = ds['test'].to_pandas()

In [8]:
import re
def clean_text(text: str) -> str:
    text = text.lower()
    text = text.replace("\n", " ")
    text = re.sub(r'""', '"', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub('\s+', ' ', text).strip()
    return text

In [9]:
%%time
df_train['cleaned_text'] = df_train['text'].apply(clean_text)
df_test['cleaned_text'] = df_test['text'].apply(clean_text)

CPU times: user 777 ms, sys: 7.03 ms, total: 784 ms
Wall time: 784 ms


## 2. Diversity Metrics

In [10]:
from softprompt.metrics.diversity import (
    vocabulary_size,
    distinct_n,
    average_pairwise_similarity,
    average_pairwise_similarity_by_class,
    inter_sample_ngram_freq
)

In [11]:
labels = y_train.copy()
texts = df_train['cleaned_text'].to_list()
embeddings = X_train.copy()
assert len(texts) == labels.shape[0]
assert len(texts) == embeddings.shape[0]

In [12]:
vocab_size = vocabulary_size(texts)
print(f"Vocab: {vocab_size:>7,}")

Vocab: 102,166


In [13]:
distinct_2 = distinct_n(texts, n=2)
print(f"Distinct-2: {distinct_2:.4f}")

Distinct-2: 0.2910


In [14]:
n = 20_000  # kernel crashes with full size
embeddings = embeddings[:n]
labels = labels[:n]

In [15]:
%%time
inter_sim, intra_sim = average_pairwise_similarity_by_class(
    embeddings, labels
)
print(f"Inter-class APS: {inter_sim:.4f}\n",
      f"Intra-class APS: {intra_sim:.4f}")

Inter-class APS: 0.1587
 Intra-class APS: 0.2293
CPU times: user 36.6 s, sys: 11.7 s, total: 48.2 s
Wall time: 3.5 s


In [16]:
%%time
avg_ps = average_pairwise_similarity(embeddings)
print(f"Avg Pairwise Similarity: {avg_ps:.4f}")

Avg Pairwise Similarity: 0.1764
CPU times: user 35.2 s, sys: 11.9 s, total: 47.2 s
Wall time: 3.41 s


## 3. Oracle Performance

In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [9]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import f1_score, accuracy_score

def evaluate_lg(X_train: np.ndarray,
                y_train: np.ndarray,
                X_test: np.ndarray,
                y_test: np.ndarray,
                subsample_size: int = 1000,
                bootstrap: bool = True,  # sampling with replacement
                n_trials: int = 30,
                n_jobs=16,) -> dict[str, tuple[float, float]]:

    assert X_train.shape[0] == len(y_train)

    train_acc_array = np.empty(n_trials)
    test_acc_array = np.empty_like(train_acc_array)
    train_f1_array = np.empty(n_trials)
    test_f1_array = np.empty_like(train_f1_array)
    
    original_idx = np.arange(X_train.shape[0])  # [0, 1, ..., len(X_train)]

    for i in range(n_trials):
        
        # get indices to use for training
        rng = np.random.default_rng(42+i)
        if bootstrap:
            use_idx = rng.choice(original_idx, size=subsample_size, replace=True)
        else:
            shuffled_idx = rng.permutation(original_idx)
            use_idx = shuffled_idx[:subsample_size]

        # fit model
        lg = LogisticRegressionCV(Cs=10, cv=5, penalty='l2',
                                  solver='lbfgs', max_iter=1000, n_jobs=n_jobs,
                                  random_state=42+i, scoring='neg_log_loss')
        lg.fit(X_train[use_idx], y_train[use_idx]);

        # evaluate (simple accuracy)
        train_acc_array[i] = accuracy_score(y_train[use_idx], lg.predict(X_train[use_idx]))
        test_acc_array[i] = accuracy_score(y_test, lg.predict(X_test))
        
        # evaluate (f1)
        train_f1_array[i] = \
            f1_score(y_train[use_idx], lg.predict(X_train[use_idx]), average='macro')
        test_f1_array[i] = \
            f1_score(y_test, lg.predict(X_test), average='macro')

    return {
        'train_accuracy': (train_acc_array.mean(), train_acc_array.std(ddof=1)),
        'test_accuracy': (test_acc_array.mean(), test_acc_array.std(ddof=1)),
        'train_f1': (train_f1_array.mean(), train_f1_array.std(ddof=1)),
        'test_f1': (test_f1_array.mean(), test_f1_array.std(ddof=1)),
    }

In [12]:
size_to_acc = {}

ratios = (0.01, 0.05, 0.10, 0.25, 0.50, 1.00)
subsample_sizes = [int(len(X_train) * r) for r in ratios]

for i, subsample_size in enumerate(subsample_sizes):

    print(f">> Sample size: {subsample_size:>6,}")

    eval_result = evaluate_lg(
        X_train, y_train, X_test, y_test,
        subsample_size=subsample_size,
        bootstrap=True,
        n_trials=30,
        n_jobs=16
    )

    train_acc, test_acc = \
        eval_result['train_accuracy'], eval_result['test_accuracy']
    train_f1, test_f1 = \
        eval_result['train_f1'], eval_result['test_f1']

    size_to_acc[subsample_size] = eval_result

    print("\t Train acc = {:.4f} ({:.4f})".format(*train_acc))
    print("\t  Test acc = {:.4f} ({:.4f})".format(*test_acc))
    print("\t Train  f1 = {:.4f} ({:.4f})".format(*train_f1))
    print("\t  Test  f1 = {:.4f} ({:.4f})".format(*test_f1))

>> Sample size:  1,200


	 Train acc = 0.9896 (0.0182)
	  Test acc = 0.8906 (0.0050)
	 Train  f1 = 0.9897 (0.0182)
	  Test  f1 = 0.8904 (0.0050)
>> Sample size:  6,000
	 Train acc = 0.9436 (0.0030)
	  Test acc = 0.9071 (0.0014)
	 Train  f1 = 0.9435 (0.0030)
	  Test  f1 = 0.9070 (0.0014)
>> Sample size: 12,000
	 Train acc = 0.9386 (0.0025)
	  Test acc = 0.9106 (0.0013)
	 Train  f1 = 0.9385 (0.0025)
	  Test  f1 = 0.9105 (0.0013)
>> Sample size: 30,000
	 Train acc = 0.9342 (0.0017)
	  Test acc = 0.9154 (0.0013)
	 Train  f1 = 0.9341 (0.0017)
	  Test  f1 = 0.9152 (0.0013)
>> Sample size: 60,000
	 Train acc = 0.9347 (0.0038)
	  Test acc = 0.9164 (0.0022)
	 Train  f1 = 0.9346 (0.0038)
	  Test  f1 = 0.9162 (0.0022)
>> Sample size: 120,000
	 Train acc = 0.9236 (0.0018)
	  Test acc = 0.9175 (0.0011)
	 Train  f1 = 0.9235 (0.0018)
	  Test  f1 = 0.9173 (0.0011)
